In [0]:
from pyspark.sql import functions as F

VOLUME_PATH = "/Volumes/food_delivery_catalog/source/food_delivery_files/"

df_restaurants = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .csv(VOLUME_PATH + "restaurants_initial.csv")
)

df_restaurant_changes = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .csv(VOLUME_PATH + "restaurants_changes.csv")
)

print("INITIAL RESTAURANTS:", df_restaurants.count())
print("RESTAURANT CHANGES:", df_restaurant_changes.count())

display(df_restaurants)
display(df_restaurant_changes)

INITIAL RESTAURANTS: 8
RESTAURANT CHANGES: 2


restaurant_id,restaurant_name,restaurant_city,primary_cuisine,service_status
RST001,Restaurant 1,Bengaluru,INDIAN,ACTIVE
RST002,Restaurant 2,Mumbai,CHINESE,ACTIVE
RST003,Restaurant 3,Kochi,CONTINENTAL,ACTIVE
RST004,Restaurant 4,Delhi,INDIAN,ACTIVE
RST005,Restaurant 5,Coimbatore,CHINESE,ACTIVE
RST006,Restaurant 6,Bengaluru,CONTINENTAL,ACTIVE
RST007,Restaurant 7,Chennai,INDIAN,ACTIVE
RST008,Restaurant 8,Hyderabad,CHINESE,ACTIVE


restaurant_id,restaurant_name,restaurant_city,primary_cuisine,service_status,effective_date
RST003,Restaurant 3,Kochi,CONTINENTAL,ACTIVE,2026-10-20
RST006,Restaurant 6,Bengaluru,CONTINENTAL,INACTIVE,2026-10-22


In [0]:

from pyspark.sql import functions as F
from pyspark.sql.window import Window

CATALOG = "food_delivery_catalog"
GOLD_SCHEMA = "gold"

DIM_RESTAURANT = f"{CATALOG}.{GOLD_SCHEMA}.dim_restaurant"

# Read restaurant master data
df_initial = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .csv(VOLUME_PATH + "restaurants_initial.csv")
)

df_changes = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .csv(VOLUME_PATH + "restaurants_changes.csv")
)

# Standardize initial data
df_initial = (
    df_initial
    .withColumn("restaurant_id", F.trim("restaurant_id"))
    .withColumn("restaurant_name", F.initcap(F.trim("restaurant_name")))
    .withColumn("restaurant_city", F.initcap(F.trim("restaurant_city")))
    .withColumn("primary_cuisine", F.upper(F.trim("primary_cuisine")))
    .withColumn("service_status", F.upper(F.trim("service_status")))
)

# Standardize changes
df_changes = (
    df_changes
    .withColumn("restaurant_id", F.trim("restaurant_id"))
    .withColumn("restaurant_name", F.initcap(F.trim("restaurant_name")))
    .withColumn("restaurant_city", F.initcap(F.trim("restaurant_city")))
    .withColumn("primary_cuisine", F.upper(F.trim("primary_cuisine")))
    .withColumn("service_status", F.upper(F.trim("service_status")))
    .withColumn("effective_date", F.to_date("effective_date"))
)

print("INITIAL COUNT:", df_initial.count())
print("CHANGE COUNT:", df_changes.count())


INITIAL COUNT: 8
CHANGE COUNT: 2


In [0]:

# Compare incoming changes against the initial master
comparison = (
    df_changes.alias("c")
    .join(
        df_initial.alias("i"),
        F.col("c.restaurant_id") == F.col("i.restaurant_id"),
        "inner"
    )
)

actual_changes = comparison.filter(
    (F.col("c.restaurant_name") != F.col("i.restaurant_name")) |
    (F.col("c.restaurant_city") != F.col("i.restaurant_city")) |
    (F.col("c.primary_cuisine") != F.col("i.primary_cuisine")) |
    (F.col("c.service_status") != F.col("i.service_status"))
)

display(
    actual_changes.select(
        F.col("c.restaurant_id"),
        F.col("i.service_status").alias("old_status"),
        F.col("c.service_status").alias("new_status"),
        F.col("c.effective_date")
    )
)


restaurant_id,old_status,new_status,effective_date
RST006,ACTIVE,INACTIVE,2026-10-22


In [0]:

# Identify restaurant IDs with genuine changes
changed_ids = actual_changes.select(
    F.col("c.restaurant_id").alias("restaurant_id")
).distinct()

# Initial records without changes remain current
unchanged_records = (
    df_initial
    .join(changed_ids, "restaurant_id", "left_anti")
    .withColumn("valid_from", F.to_date(F.lit("1900-01-01")))
    .withColumn("valid_to", F.to_date(F.lit("9999-12-31")))
    .withColumn("is_current", F.lit(True))
)

# Close old versions for changed restaurants
old_versions = (
    actual_changes
    .select(
        F.col("i.restaurant_id").alias("restaurant_id"),
        F.col("i.restaurant_name").alias("restaurant_name"),
        F.col("i.restaurant_city").alias("restaurant_city"),
        F.col("i.primary_cuisine").alias("primary_cuisine"),
        F.col("i.service_status").alias("service_status"),
        F.to_date(F.lit("1900-01-01")).alias("valid_from"),
        F.date_sub(F.col("c.effective_date"), 1).alias("valid_to"),
        F.lit(False).alias("is_current")
    )
)

# Create new versions from actual changes
new_versions = (
    actual_changes
    .select(
        F.col("c.restaurant_id").alias("restaurant_id"),
        F.col("c.restaurant_name").alias("restaurant_name"),
        F.col("c.restaurant_city").alias("restaurant_city"),
        F.col("c.primary_cuisine").alias("primary_cuisine"),
        F.col("c.service_status").alias("service_status"),
        F.col("c.effective_date").alias("valid_from"),
        F.to_date(F.lit("9999-12-31")).alias("valid_to"),
        F.lit(True).alias("is_current")
    )
)

# Combine all historical and current versions
df_restaurant_history = (
    unchanged_records
    .unionByName(old_versions)
    .unionByName(new_versions)
)

# Generate deterministic surrogate keys
sk_window = Window.orderBy("restaurant_id", "valid_from")

df_restaurant_history = (
    df_restaurant_history
    .withColumn(
        "restaurant_sk",
        F.row_number().over(sk_window)
    )
    .select(
        "restaurant_sk",
        "restaurant_id",
        "restaurant_name",
        "restaurant_city",
        "primary_cuisine",
        "service_status",
        "valid_from",
        "valid_to",
        "is_current"
    )
)

display(
    df_restaurant_history.orderBy("restaurant_id", "valid_from")
)


/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1155: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


restaurant_sk,restaurant_id,restaurant_name,restaurant_city,primary_cuisine,service_status,valid_from,valid_to,is_current
1,RST001,Restaurant 1,Bengaluru,INDIAN,ACTIVE,1900-01-01,9999-12-31,true
2,RST002,Restaurant 2,Mumbai,CHINESE,ACTIVE,1900-01-01,9999-12-31,true
3,RST003,Restaurant 3,Kochi,CONTINENTAL,ACTIVE,1900-01-01,9999-12-31,true
4,RST004,Restaurant 4,Delhi,INDIAN,ACTIVE,1900-01-01,9999-12-31,true
5,RST005,Restaurant 5,Coimbatore,CHINESE,ACTIVE,1900-01-01,9999-12-31,true
6,RST006,Restaurant 6,Bengaluru,CONTINENTAL,ACTIVE,1900-01-01,2026-10-21,false
7,RST006,Restaurant 6,Bengaluru,CONTINENTAL,INACTIVE,2026-10-22,9999-12-31,true
8,RST007,Restaurant 7,Chennai,INDIAN,ACTIVE,1900-01-01,9999-12-31,true
9,RST008,Restaurant 8,Hyderabad,CHINESE,ACTIVE,1900-01-01,9999-12-31,true


In [0]:

(
    df_restaurant_history.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(DIM_RESTAURANT)
)

print("DIM_RESTAURANT CREATED")

print(
    "TOTAL VERSIONS:",
    spark.table(DIM_RESTAURANT).count()
)

print(
    "CURRENT RESTAURANTS:",
    spark.table(DIM_RESTAURANT)
    .filter("is_current = true")
    .count()
)


/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1155: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


DIM_RESTAURANT CREATED
TOTAL VERSIONS: 9
CURRENT RESTAURANTS: 8


In [0]:
%sql
SELECT
    restaurant_id,
    restaurant_name,
    service_status,
    valid_from,
    valid_to,
    is_current
FROM food_delivery_catalog.gold.dim_restaurant
WHERE restaurant_id IN ('RST003', 'RST006')
ORDER BY restaurant_id, valid_from;

restaurant_id,restaurant_name,service_status,valid_from,valid_to,is_current
RST003,Restaurant 3,ACTIVE,1900-01-01,9999-12-31,true
RST006,Restaurant 6,ACTIVE,1900-01-01,2026-10-21,false
RST006,Restaurant 6,INACTIVE,2026-10-22,9999-12-31,true


In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

CATALOG = "food_delivery_catalog"
GOLD_SCHEMA = "gold"

SILVER_TABLE = f"{CATALOG}.silver.orders_cleaned"
RESTAURANT_DIM = f"{CATALOG}.gold.dim_restaurant"

df_orders = spark.table(SILVER_TABLE)
df_restaurant = spark.table(RESTAURANT_DIM)

print("SILVER ORDERS:", df_orders.count())
print("RESTAURANT VERSIONS:", df_restaurant.count())

SILVER ORDERS: 83
RESTAURANT VERSIONS: 9


In [0]:
df_city = (
    df_orders
    .select(F.col("city").alias("city_name"))
    .distinct()
    .withColumn(
        "city_sk",
        F.row_number().over(
            Window.orderBy("city_name")
        )
    )
    .select("city_sk", "city_name")
)

df_city.write.format("delta").mode("overwrite").option(
    "overwriteSchema", "true"
).saveAsTable(f"{CATALOG}.gold.dim_city")

print("CITY DIMENSION COUNT:", df_city.count())
display(df_city)

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1155: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


CITY DIMENSION COUNT: 8


city_sk,city_name
1,Bengaluru
2,Chennai
3,Coimbatore
4,Delhi
5,Hyderabad
6,Kochi
7,Mumbai
8,Pune


In [0]:
df_category = (
    df_orders
    .select(F.col("food_category").alias("category_name"))
    .distinct()
    .withColumn(
        "food_category_sk",
        F.row_number().over(
            Window.orderBy("category_name")
        )
    )
    .select("food_category_sk", "category_name")
)

df_category.write.format("delta").mode("overwrite").option(
    "overwriteSchema", "true"
).saveAsTable(f"{CATALOG}.gold.dim_food_category")

print("CATEGORY COUNT:", df_category.count())
display(df_category)

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1155: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


CATEGORY COUNT: 6


/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1155: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


food_category_sk,category_name
1,CHINESE
2,CONTINENTAL
3,INDIAN
4,ITALIAN
5,MEXICAN
6,THAI


/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1155: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


In [0]:
df_date = (
    df_orders
    .select("order_date")
    .distinct()
    .withColumn(
        "date_sk",
        F.date_format("order_date", "yyyyMMdd").cast("INT")
    )
    .withColumn("year", F.year("order_date"))
    .withColumn("month", F.month("order_date"))
    .withColumn("month_name", F.date_format("order_date", "MMMM"))
    .withColumn("quarter", F.quarter("order_date"))
    .withColumn("day", F.dayofmonth("order_date"))
    .withColumn("day_name", F.date_format("order_date", "EEEE"))
)

df_date.write.format("delta").mode("overwrite").option(
    "overwriteSchema", "true"
).saveAsTable(f"{CATALOG}.gold.dim_date")

print("DATE DIMENSION COUNT:", df_date.count())
display(df_date)

DATE DIMENSION COUNT: 31


order_date,date_sk,year,month,month_name,quarter,day,day_name
2026-10-22,20261022,2026,10,October,4,22,Thursday
2026-10-25,20261025,2026,10,October,4,25,Sunday
2026-10-17,20261017,2026,10,October,4,17,Saturday
2026-10-10,20261010,2026,10,October,4,10,Saturday
2026-10-26,20261026,2026,10,October,4,26,Monday
2026-10-28,20261028,2026,10,October,4,28,Wednesday
2026-10-12,20261012,2026,10,October,4,12,Monday
2026-10-03,20261003,2026,10,October,4,3,Saturday
2026-10-13,20261013,2026,10,October,4,13,Tuesday
2026-10-08,20261008,2026,10,October,4,8,Thursday


In [0]:
# Alias the tables
o = df_orders.alias("o")
r = df_restaurant.alias("r")
c = spark.table(f"{CATALOG}.gold.dim_city").alias("c")
fc = spark.table(f"{CATALOG}.gold.dim_food_category").alias("fc")
d = spark.table(f"{CATALOG}.gold.dim_date").alias("d")

# Join orders to the restaurant version valid on order_date
df_fact = (
    o.join(
        r,
        (
            (F.col("o.restaurant_name") == F.col("r.restaurant_name")) &
            (F.col("o.order_date") >= F.col("r.valid_from")) &
            (F.col("o.order_date") <= F.col("r.valid_to"))
        ),
        "left"
    )
    .join(
        c,
        F.col("o.city") == F.col("c.city_name"),
        "left"
    )
    .join(
        fc,
        F.col("o.food_category") == F.col("fc.category_name"),
        "left"
    )
    .join(
        d,
        F.col("o.order_date") == F.col("d.order_date"),
        "left"
    )
)

# Select fact table columns
df_fact = df_fact.select(
    F.col("o.order_id"),
    F.col("d.date_sk"),
    F.col("r.restaurant_sk"),
    F.col("c.city_sk"),
    F.col("fc.food_category_sk"),
    F.col("o.customer_id"),
    F.col("o.quantity"),
    F.col("o.item_price"),
    F.col("o.delivery_fee"),
    F.col("o.discount"),
    F.col("o.order_amount"),
    F.col("o.order_status"),
    F.col("o.batch_id"),
    F.col("o.ingestion_timestamp")
)

print("FACT ROW COUNT:", df_fact.count())
display(df_fact.limit(10))

FACT ROW COUNT: 83


order_id,date_sk,restaurant_sk,city_sk,food_category_sk,customer_id,quantity,item_price,delivery_fee,discount,order_amount,order_status,batch_id,ingestion_timestamp
ORD0001,20261005,2,7,1,C001,2,375.00,90.00,20.00,820.00,DELIVERED,BATCH_001,2026-11-26T05:10:44.030Z
ORD0002,20261006,3,6,2,C002,3,550.00,130.00,40.00,1740.00,CANCELLED,BATCH_001,2026-11-26T05:10:44.030Z
ORD0003,20261007,4,4,3,C003,4,725.00,170.00,60.00,3010.00,PLACED,BATCH_001,2026-11-26T05:10:44.030Z
ORD0004,20261008,5,3,1,C004,5,900.00,210.00,0.00,4710.00,DELIVERED,BATCH_001,2026-11-26T05:10:44.030Z
ORD0006,20261010,8,2,3,C006,2,1250.00,290.00,40.00,2750.00,PLACED,BATCH_001,2026-11-26T05:10:44.030Z
ORD0007,20261011,9,5,1,C007,3,1425.00,50.00,60.00,4265.00,DELIVERED,BATCH_001,2026-11-26T05:10:44.030Z
ORD0008,20261012,1,8,2,C008,4,200.00,90.00,0.00,890.00,CANCELLED,BATCH_001,2026-11-26T05:10:44.030Z
ORD0009,20261013,2,7,3,C009,5,375.00,130.00,20.00,1985.00,PLACED,BATCH_001,2026-11-26T05:10:44.030Z
ORD0010,20261014,3,6,1,C010,1,550.00,170.00,40.00,680.00,DELIVERED,BATCH_001,2026-11-26T05:10:44.030Z
ORD0005,20261009,6,1,2,C005,1,1075.00,250.00,20.00,1305.00,CANCELLED,BATCH_001,2026-11-26T05:10:44.030Z


In [0]:
FACT_TABLE = f"{CATALOG}.gold.fact_orders"

(
    df_fact.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(FACT_TABLE)
)

print("FACT TABLE CREATED")
print("FACT COUNT:", spark.table(FACT_TABLE).count())

FACT TABLE CREATED
FACT COUNT: 83


1.Which are the top 3 restaurants based on revenue generated from successfully delivered orders?

In [0]:
%sql
SELECT
    r.restaurant_name,
    COUNT(f.order_id) AS delivered_orders_count,
    ROUND(SUM(f.order_amount), 2) AS total_order_amount
FROM food_delivery_catalog.gold.fact_orders f
JOIN food_delivery_catalog.gold.dim_restaurant r
    ON f.restaurant_sk = r.restaurant_sk
WHERE f.order_status = 'DELIVERED'
GROUP BY
    r.restaurant_name
ORDER BY
    total_order_amount DESC
LIMIT 3;

restaurant_name,delivered_orders_count,total_order_amount
Restaurant 5,2,8360.00
Restaurant 8,4,8200.00
Restaurant 6,3,5555.00


2.Which cities generate the highest delivered-order revenue?

Business question: Which geographical markets contribute the most revenue from completed orders?

In [0]:
%sql
SELECT
    c.city_name,
    COUNT(f.order_id) AS delivered_orders,
    ROUND(SUM(f.order_amount), 2) AS total_revenue
FROM food_delivery_catalog.gold.fact_orders f
JOIN food_delivery_catalog.gold.dim_city c
    ON f.city_sk = c.city_sk
WHERE f.order_status = 'DELIVERED'
GROUP BY c.city_name
ORDER BY total_revenue DESC;

city_name,delivered_orders,total_revenue
Coimbatore,4,8800.00
Hyderabad,3,7720.00
Bengaluru,3,5555.00
Delhi,3,4790.00
Chennai,1,3800.00
Kochi,2,3680.00
Mumbai,3,1870.00
Pune,3,1195.00


5. Weekly revenue trend
Business question: How does delivered revenue change week by week, and which week generates the highest revenue?

In [0]:
%sql
WITH weekly_revenue AS (
    SELECT
        YEAR(d.order_date) AS order_year,
        WEEKOFYEAR(d.order_date) AS week_number,
        MIN(d.order_date) AS week_start_date,
        SUM(order_amount) AS total_revenue
    FROM food_delivery_catalog.gold.fact_orders f
    JOIN food_delivery_catalog.gold.dim_date d
        ON f.date_sk = d.date_sk
    WHERE order_status = 'DELIVERED'
    GROUP BY
        YEAR(d.order_date),
        WEEKOFYEAR(d.order_date)
),

revenue_comparison AS (
    SELECT
        *,
        LAG(total_revenue) OVER (
            ORDER BY order_year, week_number
        ) AS previous_week_revenue
    FROM weekly_revenue
)

SELECT
    order_year,
    week_number,
    week_start_date,
    ROUND(total_revenue, 2) AS current_week_revenue,
    ROUND(previous_week_revenue, 2) AS previous_week_revenue,
    ROUND(
        100.0 * (total_revenue - previous_week_revenue)
        / NULLIF(previous_week_revenue, 0),
        2
    ) AS revenue_growth_percentage
FROM revenue_comparison
ORDER BY order_year, week_number;

order_year,week_number,week_start_date,current_week_revenue,previous_week_revenue,revenue_growth_percentage
2026,40,2026-10-01,4215.00,null,null
2026,41,2026-10-05,16970.00,4215.00,302.61
2026,42,2026-10-14,6680.00,16970.00,-60.64
2026,43,2026-10-20,5045.00,6680.00,-24.48
2026,44,2026-10-26,4500.00,5045.00,-10.80


Which restaurants generate the highest gross sales, and how much revenue do they lose through discounts?

In [0]:
%sql
SELECT
    r.restaurant_name,

    COUNT(f.order_id) AS total_orders,

    ROUND(
        SUM(f.quantity * f.item_price + f.delivery_fee), 2
    ) AS gross_sales,

    ROUND(SUM(f.discount), 2) AS total_discounts,

    ROUND(SUM(f.order_amount), 2) AS net_revenue,

    ROUND(
        100.0 * SUM(f.discount) /
        NULLIF(SUM(f.quantity * f.item_price + f.delivery_fee), 0),
        2
    ) AS discount_percentage

FROM food_delivery_catalog.gold.fact_orders f

JOIN food_delivery_catalog.gold.dim_restaurant r
    ON f.restaurant_sk = r.restaurant_sk

WHERE f.order_status = 'DELIVERED'

GROUP BY r.restaurant_name

ORDER BY net_revenue DESC;

restaurant_name,total_orders,gross_sales,total_discounts,net_revenue,discount_percentage
Restaurant 5,2,8360.00,0.00,8360.00,0.00
Restaurant 8,4,8355.00,155.00,8200.00,1.86
Restaurant 6,3,5640.00,85.00,5555.00,1.51
Restaurant 4,2,4410.00,100.00,4310.00,2.27
Restaurant 3,4,4250.00,130.00,4120.00,3.06
Restaurant 7,2,4055.00,40.00,4015.00,0.99
Restaurant 2,3,1920.00,50.00,1870.00,2.60
Restaurant 1,2,980.00,0.00,980.00,0.00
